In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 63.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found exi

In [2]:
import pandas as pd
import os
import numpy as np
import shutil
import yaml
import matplotlib.pyplot as plt
import random
import cv2

from sklearn import model_selection
from tqdm import tqdm
from glob import glob

In [3]:
size = 512
TRAIN_LABELS_PATH = './vinbigdata/labels/train'
VAL_LABELS_PATH = './vinbigdata/labels/val'
TRAIN_IMAGES_PATH = './vinbigdata/images/train' #12000
VAL_IMAGES_PATH = './vinbigdata/images/val' #3000
External_DIR = f'../input/vinbigdata-{size}-image-dataset/vinbigdata/train' # 15000
os.makedirs(TRAIN_LABELS_PATH, exist_ok = True)
os.makedirs(VAL_LABELS_PATH, exist_ok = True)
os.makedirs(TRAIN_IMAGES_PATH, exist_ok = True)
os.makedirs(VAL_IMAGES_PATH, exist_ok = True)

In [4]:
original_df = pd.read_csv('../input/vinbigdata-chest-xray-abnormalities-detection/train.csv')
number_of_imageids = len(original_df['image_id'].values)
print(f'Total number of image_ids (train + validation) {number_of_imageids}')

number_of_images = len(os.listdir('../input/vinbigdata-chest-xray-abnormalities-detection/train'))
print(f'Total number of images (train + validation) {number_of_images}')

number_of_labels = len(os.listdir('../input/vinbigdata-yolo-labels-dataset/labels'))
print(f'Total number of labels (train + validation) {number_of_labels}')

Total number of image_ids (train + validation) 67914
Total number of images (train + validation) 15000
Total number of labels (train + validation) 15000


In [5]:
import os
import cv2
import numpy as np
import pydicom
import multiprocessing
from tqdm import tqdm
from skimage import exposure

def dicom2array(path, voi_lut=True, fix_monochrome=True):
    dicom = pydicom.read_file(path)
    if voi_lut:
        data = apply_voi_lut(dicom.pixel_array, dicom)
    else:
        data = dicom.pixel_array
    if fix_monochrome and dicom.PhotometricInterpretation == "MONOCHROME1":
        data = np.amax(data) - data
    data = data - np.min(data)
    data = data / np.max(data)
    data = (data * 255).astype(np.uint8)
    return data

def process_image(dicom_path_output_dir):
    dicom_path, output_dir = dicom_path_output_dir
    file_name = os.path.splitext(os.path.basename(dicom_path))[0]
    image_array = dicom2array(dicom_path)
    equalized_image = exposure.equalize_hist(image_array)
    equalized_image = (equalized_image * 255).astype(np.uint8)
    cv2.imwrite(os.path.join(output_dir, f"{file_name}.jpeg"), equalized_image)

def saving_image(output_dir, dicom_path_list):
    os.makedirs(output_dir, exist_ok=True)
    dicom_path_output_dir_list = [(path, output_dir) for path in dicom_path_list]

    # Use multiprocessing Pool for parallel processing
    with multiprocessing.Pool() as pool:
        list(tqdm(pool.imap(process_image, dicom_path_output_dir_list), total=len(dicom_path_list), desc="Processing Images"))

In [6]:
df = pd.read_csv('/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train.csv')
number_of_images = len(df['image_id'].values)
print(f'Total number of image ids (train + validation) {number_of_images}')

df = df[df.class_id!=14].reset_index(drop = True)
number_of_images = len(df['image_id'].values)
print(f'Total number of image ids after dropping normal images (train + validation) {number_of_images}')

df.head()

Total number of image ids (train + validation) 67914
Total number of image ids after dropping normal images (train + validation) 36096


,image_id,class_name,class_id,rad_id,x_min,y_min,x_max,y_max
0,9a5094b2563a1ef3ff50dc5c7ff71345,Cardiomegaly,3,R10,691.0,1375.0,1653.0,1831.0
1,051132a778e61a86eb147c7c6f564dfe,Aortic enlargement,0,R10,1264.0,743.0,1611.0,1019.0
2,1c32170b4af4ce1a3030eb8167753b06,Pleural thickening,11,R9,627.0,357.0,947.0,433.0
3,0c7a38f293d5f5e4846aa4ca6db4daf1,ILD,5,R17,1347.0,245.0,2188.0,2169.0
4,47ed17dcb2cbeec15182ed335a8b5a9e,Nodule/Mass,8,R9,557.0,2352.0,675.0,2484.0


In [7]:
df = df.drop(columns=['class_name', 'rad_id', 'x_min', 'x_max', 'y_min', 'y_max',  'class_id']) # we only need image ids, labels are pre-made
df.head()

,image_id
0,9a5094b2563a1ef3ff50dc5c7ff71345
1,051132a778e61a86eb147c7c6f564dfe
2,1c32170b4af4ce1a3030eb8167753b06
3,0c7a38f293d5f5e4846aa4ca6db4daf1
4,47ed17dcb2cbeec15182ed335a8b5a9e


In [8]:
df_train, df_valid = model_selection.train_test_split(df, test_size=0.05, random_state=42, shuffle=True)

In [9]:
number_of_images = len(df_train['image_id'].values)
print(f'Total number of training image_ids {number_of_images}')

number_of_images = len(df_valid['image_id'].values)
print(f'Total number of validation image_ids {number_of_images}')


Total number of training image_ids 34291
Total number of validation image_ids 1805


In [10]:
# need to delete duplicate image ids, len(labels) should be equal len(df.imageids.values), 

In [11]:
print(f'Total number of training images {len(df_train.image_id.unique())}')
print(f'Total number of validation images {len(df_valid.image_id.unique())}')

Total number of training images 4394
Total number of validation images 1434


In [12]:
def preproccess_data(df, labels_path, images_path):
    for img_id in tqdm(df.image_id.unique()):
        shutil.copy(os.path.join('../input/vinbigdata-yolo-labels-dataset/labels', f"{img_id}"+'.txt'), labels_path)
        shutil.copy(os.path.join(f'/kaggle/input/vinbigdata-{size}-image-dataset/vinbigdata/train', f"{img_id}.png"), images_path)

In [13]:
preproccess_data(df_train, TRAIN_LABELS_PATH, TRAIN_IMAGES_PATH)
preproccess_data(df_valid, VAL_LABELS_PATH, VAL_IMAGES_PATH)

100%|██████████| 1434/1434 [00:03<00:00, 419.16it/s]


In [14]:
# check that data was preprocessed correctly
print(len(os.listdir(TRAIN_LABELS_PATH)))
print(len(os.listdir(TRAIN_IMAGES_PATH)))

print(len(os.listdir(VAL_LABELS_PATH)))
print(len(os.listdir(VAL_IMAGES_PATH)))

4394
4394
1434
1434


In [15]:
# credit / source https://www.kaggle.com/awsaf49/vinbigdata-cxr-ad-yolov5-14-class-train
classes = [ 'Aortic enlargement',
            'Atelectasis',
            'Calcification',
            'Cardiomegaly',
            'Consolidation',
            'ILD',
            'Infiltration',
            'Lung Opacity',
            'Nodule/Mass',
            'Other lesion',
            'Pleural effusion',
            'Pleural thickening',
            'Pneumothorax',
            'Pulmonary fibrosis']

data = dict(
    train =  '../vinbigdata/images/train',
    val   =  '../vinbigdata/images/val',
    nc    = 14,
    names = classes
    )

with open('/kaggle/working/vinbigdata.yaml', 'w') as outfile:
    yaml.dump(data, outfile, default_flow_style=False)

f = open(os.path.join( os.getcwd() , 'vinbigdata.yaml'), 'r')
print('\nyaml:')
print(f.read())


yaml:
names:
- Aortic enlargement
- Atelectasis
- Calcification
- Cardiomegaly
- Consolidation
- ILD
- Infiltration
- Lung Opacity
- Nodule/Mass
- Other lesion
- Pleural effusion
- Pleural thickening
- Pneumothorax
- Pulmonary fibrosis
nc: 14
train: ../vinbigdata/images/train
val: ../vinbigdata/images/val



In [16]:
import os
from ultralytics import YOLO
import torch

# Environment setup
os.environ["WANDB_MODE"] = "dryrun"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  # Make both GPUs visible

def setup_finetuning():
    # Verify CUDA availability
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Check your GPU configuration.")
    
    print(f"⚡ Available GPUs: {torch.cuda.device_count()}")
    assert torch.cuda.device_count() >= 2, "Need at least 2 GPUs for this configuration"
    
    # Load YOUR custom pretrained model
    model_path = '/kaggle/input/modfee/yolo12s 77map.pt'
    model = YOLO(model_path)
    print(f"\n🔧 Loaded custom model from: {model_path}")
    
    # Find dataset (modify as needed)
    dataset_path = './vinbigdata.yaml'
    if not os.path.exists(dataset_path):
        print("\n🔍 Searching for dataset...")
        !find /kaggle/input -name "data.yaml"  # Search entire input directory
        for root, _, files in os.walk('/kaggle/input'):
            if 'data.yaml' in files:
                dataset_path = os.path.join(root, 'data.yaml')
                break
        else:
            raise FileNotFoundError("data.yaml not found in /kaggle/input")
    print(f"✅ Found dataset at: {dataset_path}")

    # Fine-tuning configuration
    train_args = {
        'data': dataset_path,
        'epochs': 200,  # Typically fewer epochs for fine-tuning
        'batch': 32,   # 16 per GPU
        'device': [0, 1],
        'workers': 8,  # 4 per GPU
        
        # Fine-tuning specific settings
        'optimizer': 'AdamW',  # Better for fine-tuning
        'lr0': 0.0001,  # Lower learning rate than pretraining
        'lrf': 0.01,    # Final LR = 0.000001
        'cos_lr': True,  # Smooth LR decay
        'warmup_epochs': 3,
        
        # Augmentations (moderate)
        'imgsz': 640,
        'mosaic': 0.3,  # Less aggressive than pretraining
        'fliplr': 0.3,
        
        # Performance
        'amp': True,
        'cache': 'ram',
        'resume': False,  # Important for fine-tuning
        'pretrained': False,  # Use our custom weights
        'name': 'mbvfss_finetune'
    }
    
    return model, train_args

def main():
    try:
        # Verify environment
        print("\n🔍 Hardware verification:")
        !nvidia-smi --query-gpu=name,memory.total --format=csv
        
        model, train_args = setup_finetuning()
        
        # Pre-training validation
        print("\n🔍 Running initial validation...")
        model.val(
            data=train_args['data'],
            batch=32,
            device=[0,1],
            workers=8
        )
        
        # Start fine-tuning
        print(f"\n🚀 Fine-tuning with batch={train_args['batch']} (16 per GPU)")
        model.train(**train_args)
        
        # Final validation
        print("\n🎯 Final validation:")
        model.val(
            data=train_args['data'],
            batch=32,
            device=[0,1],
            plots=True
        )
        
    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
        print("\n💡 Debugging tips:")
        print("1. Verify model path exists: !ls /kaggle/input/mbvfss")
        print("2. Check dataset structure: !cat {train_args['data']}")
        print("3. Try reducing batch size if OOM occurs")

if __name__ == "__main__":
    main()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🔍 Hardware verification:
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
⚡ Available GPUs: 2

🔧 Loaded custom model from: /kaggle/input/modfee/yolo12s 77map.pt
✅ Found dataset at: ./vinbigdata.yaml

🔍 Running initial validation...
Ultralytics 8.3.161 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                        CUDA:1 (Tesla T4, 15095MiB)
YOLOv12s summary (fused): 159 layers, 9,236,298 parameters, 0 gradients, 21.2 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 18.8MB/s]

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2403.8±669.0 MB/s, size: 135.0 KB)



val: Scanning /kaggle/working/vinbigdata/labels/val... 1434 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1434/1434 [00:02<00:00, 679.63it/s]

val: New cache created: /kaggle/working/vinbigdata/labels/val.cache


...

                   all       1434      14888      0.943      0.704      0.801      0.677
    Aortic enlargement       1000       2282      0.992       0.59      0.765      0.681
           Atelectasis         85        128      0.965      0.859      0.906       0.79
         Calcification        192        443      0.895      0.617      0.749       0.56
          Cardiomegaly        745       1725      0.994      0.562      0.733      0.664
         Consolidation        170        288      0.926      0.784      0.855      0.774
                   ILD        161        419       0.98      0.809      0.868      0.799
          Infiltration        263        564      0.955      0.787      0.852      0.755
          Lung Opacity        559       1210      0.956      0.769      0.859      0.737
           Nodule/Mass        344       1368      0.868      0.541      0.647      0.465
          Other lesion        445        989      0.945      0.783      0.865      0.714
      Pleural effusio

val: Scanning /kaggle/working/vinbigdata/labels/val.cache... 1434 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1434/1434 [00:00<?, ?it/s]

WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.



val: Caching images (1.6GB RAM): 100%|██████████| 1434/1434 [00:03<00:00, 438.37it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 45/45 [00:32<00:00,  1.39it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       1434      14888      0.943      0.703      0.801      0.678
    Aortic enlargement       1000       2282      0.991       0.59      0.763      0.681
           Atelectasis         85        128      0.967      0.859      0.906      0.789
         Calcification        192        443      0.895      0.615       0.75      0.562
          Cardiomegaly        745       1725      0.994      0.561      0.734      0.665
         Consolidation        170        288      0.926      0.784      0.855      0.771
                   ILD        161        419       0.98      0.808      0.868        0.8
          Infiltration        263        564      0.955      0.788      0.852      0.757
          Lung Opacity        559       1210      0.958      0.768      0.859      0.738
           Nodule/Mass        344       1368      0.868       0.54      0.645      0.465
          Other lesion        445        989      0.945      0.782      0.866      0.716
      Pleural effusio

In [17]:
test_df = pd.read_csv(f'/kaggle/input/vinbigdata-{size}-image-dataset/vinbigdata/test.csv')

In [18]:
test_dir = f'/kaggle/input/vinbigdata-{size}-image-dataset/vinbigdata/test'
os.listdir('/kaggle/working/runs/detect/train/weights/')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/runs/detect/train/weights/'

In [ ]:
weights_dir = "/kaggle/working/runs/detect/train/weights/best.pt"  # Trọng số mô hình đã huấn luyện

model = YOLO(weights_dir)

results = model.predict(
    source=test_dir,
    imgsz=640,
    conf=0.005,
    iou=0.45,
    save_txt=True,
    save_conf=True,
    project="/kaggle/working/runs/detect",  
    name="exp",                            
    exist_ok=True,                         
    save=True,
    verbose=False# Lưu ảnh với bounding box
)

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

# 1. Point to your YOLO output directory
output_dir = "/kaggle/working/runs/detect/exp"

# 2. Gather all .jpg/.png files
images = [
    os.path.join(output_dir, f)
    for f in sorted(os.listdir(output_dir))
    if f.lower().endswith((".jpg", ".png"))
]

# 3. Display the first 4 results in a 2×2 grid
plt.figure(figsize=(12, 12))
for i, img_path in enumerate(images[:4]):
    img = Image.open(img_path)
    plt.subplot(2, 2, i + 1)
    plt.imshow(img)
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# ─── Cell: Validation-Batch Visualization ───
# Author: mo

import os
import re
from typing import List, Tuple, Optional
from PIL import Image
import matplotlib.pyplot as plt

def get_val_batch_pairs(dir_path: str) -> List[Tuple[str, str]]:
    label_regex = re.compile(r"val_batch(\d+)_labels\.\w+$")
    pred_regex  = re.compile(r"val_batch(\d+)_pred\.\w+$")
    labels, preds = {}, {}

    for root, _, files in os.walk(dir_path):
        for fname in files:
            full = os.path.join(root, fname)
            m = label_regex.match(fname)
            if m:
                labels[int(m.group(1))] = full
            m = pred_regex.match(fname)
            if m:
                preds[int(m.group(1))] = full

    idxs = sorted(set(labels) & set(preds))
    if not idxs:
        raise ValueError(f"No pairs found in {dir_path}")
    return [(labels[i], preds[i]) for i in idxs]

def show_val_batch_comparisons(
    dir_path: str,
    max_batches: Optional[int] = None,
    save_path: Optional[str] = None
) -> None:
    pairs = get_val_batch_pairs(dir_path)
    if max_batches is not None:
        pairs = pairs[:max_batches]
    n = len(pairs)
    fig, axs = plt.subplots(n, 2, figsize=(12, 4*n), squeeze=False)

    for i, (lab, pred) in enumerate(pairs):
        for j, (path, title) in enumerate([(lab, "Labels"), (pred, "Predictions")]):
            axs[i][j].imshow(Image.open(path))
            axs[i][j].set_title(f"Batch {i}: {title}")
            axs[i][j].axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved to {save_path}")
    plt.show()

# ─── End of Cell ───────────────────────────


In [ ]:
# Show up to the first 6 validation batches:
show_val_batch_comparisons(
    dir_path="/kaggle/working/runs/detect/train",
    max_batches=6
)


In [ ]:
# ─── Cell: Validation-Batch Visualization (Enlarged) ───
# Author: mo

import os
import re
from typing import List, Tuple, Optional
from PIL import Image
import matplotlib.pyplot as plt

def get_val_batch_pairs(dir_path: str) -> List[Tuple[str, str]]:
    label_regex = re.compile(r"val_batch(\d+)_labels\.\w+$")
    pred_regex  = re.compile(r"val_batch(\d+)_pred\.\w+$")
    labels, preds = {}, {}

    for root, _, files in os.walk(dir_path):
        for fname in files:
            full = os.path.join(root, fname)
            m = label_regex.match(fname)
            if m:
                labels[int(m.group(1))] = full
            m = pred_regex.match(fname)
            if m:
                preds[int(m.group(1))] = full

    idxs = sorted(set(labels) & set(preds))
    if not idxs:
        raise ValueError(f"No pairs found in {dir_path}")
    return [(labels[i], preds[i]) for i in idxs]

def show_val_batch_comparisons(
    dir_path: str,
    max_batches: Optional[int] = None,
    save_path: Optional[str] = None,
    dpi: int = 150,
    width_per_image: float = 8.0,
    height_per_image: float = 6.0
) -> None:
    """
    Display enlarged validation batch label vs. prediction comparisons.

    Args:
        dir_path (str): Directory containing validation batch images.
        max_batches (Optional[int]): Limit display to first N batches.
        save_path (Optional[str]): Filepath to save the figure (optional).
        dpi (int): Dots-per-inch of the resulting figure.
        width_per_image (float): Width in inches of each subplot.
        height_per_image (float): Height in inches of each subplot.
    """
    pairs = get_val_batch_pairs(dir_path)
    if max_batches is not None:
        pairs = pairs[:max_batches]
    n = len(pairs)
    if n == 0:
        raise ValueError("No batches to display after filtering.")

    # Compute total figure size
    fig_width  = width_per_image * 2       # two columns
    fig_height = height_per_image * n      # one row per batch

    fig, axs = plt.subplots(
        nrows=n,
        ncols=2,
        figsize=(fig_width, fig_height),
        squeeze=False,
        dpi=dpi
    )

    for i, (lab, pred) in enumerate(pairs):
        for j, (path, title) in enumerate([(lab, "Labels"), (pred, "Predictions")]):
            axs[i][j].imshow(Image.open(path))
            axs[i][j].set_title(f"Batch {i}: {title}", fontsize=16)
            axs[i][j].axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved comparison figure to: {save_path}")
    plt.show()

# ─── End of Cell ───────────────────────────


In [ ]:
show_val_batch_comparisons(
    dir_path="runs/detect/train",
    max_batches=6,          # for example
    dpi=200,                # even sharper
    width_per_image=10.0,   # wider
    height_per_image=7.5    # taller
)
